# New Zealand DEM remake

See the original notebook [here](https://nbviewer.org/github/royalosyin/Work-with-DEM-data-using-Python-from-Simple-to-Complicated/blob/master/Sup03-Ridgelines%20Map%20of%20DEM.ipynb).

In [1]:
import pandas as pd

from lets_plot import *

In [2]:
LetsPlot.setup_html()

In [3]:
def dataset_array_to_dataframe(dataset_array):
    df = pd.DataFrame.from_records([
        (j, i, a)
        for i, r in enumerate(dataset_array)
        for j, a in enumerate(r)
    ], columns=["x", "y", "h"])
    return df

def process_rows(df, *, min_h, step_x=1, step_y=1):
    def add_tails_to_row(subdf, y):
        xs = [x for x in range(subdf['x'].min() - step_x, subdf['x'].max() + step_x + 1, step_x) if x not in subdf['x'].values]
        ys = [y] * len(xs)
        hs = [0] * len(xs)
        new_df = pd.concat([
            subdf,
            pd.DataFrame({'x': xs, 'y': ys, 'h': hs})
        ], ignore_index=True).sort_values(by='x').reset_index(drop=True)
        xs = []
        ys = []
        hs = []
        for i, (_x, _y, _h) in new_df.iterrows():
            if _h > 0:
                xs.append(_x)
                ys.append(_y)
                hs.append(_h)
            else:
                if i > 0 and new_df.iloc[i - 1]['h'] > 0:
                    xs.append(_x)
                    ys.append(_y)
                    hs.append(min_h)
                elif i < new_df.shape[0] - 1 and new_df.iloc[i + 1]['h'] > 0:
                    xs.append(_x)
                    ys.append(_y)
                    hs.append(min_h)
                else:
                    xs.append(_x)
                    ys.append(_y)
                    hs.append(0)
        return pd.DataFrame({'x': xs, 'y': ys, 'h': hs})
    return pd.concat([
        add_tails_to_row(df[df['y'] == y], y) for y in range(df['y'].min(), df['y'].max() + 1, step_y)
    ]).sort_values(by=['y', 'x']).reset_index(drop=True)

In [4]:
raw_data_array = pd.read_csv("https://raw.githubusercontent.com/JetBrains/lets-plot-docs/master/data/new_zealand.csv", header=None).to_numpy()
df = dataset_array_to_dataframe(raw_data_array)
min_h = df[df['h'] > 0].describe()['h']['min']
df = process_rows(df[df["h"] > 0], min_h=min_h, step_x=2, step_y=2)
bbox = dict(xmin=df['x'].min(), ymin=df['y'].min(), xmax=df['x'].max(), ymax=df['y'].max())
print(df.shape)
df.head()

(13767, 3)


,x,y,h
0,431.0,6.0,0.258861
1,433.0,6.0,114.518776
2,435.0,6.0,180.143860
3,437.0,6.0,0.258861
4,419.0,8.0,0.258861


In [5]:
ggplot(df) + \
    geom_area_ridges(aes("x", "y", height="h"), \
                     stat='identity', min_height=min_h, scale=.0025, \
                     color="#08519c", fill="#bdd7e7", \
                     sampling=sampling_pick(df.shape[0]), \
                     tooltips=layer_tooltips().line("height|@h").format("@h", ',.1~f'), \
                     show_legend=False) + \
    geom_text(x=bbox['xmin'] + .7 * (bbox['xmax'] - bbox['xmin']), \
              y=bbox['ymin'] + .9 * (bbox['ymax'] - bbox['ymin']), \
              label="New Zealand", size=25, family="Cinzel") + \
    scale_y_continuous(trans='reverse') + \
    ggsize(600, 600) + \
    theme_minimal() + \
    theme(axis='blank', panel_grid='blank', \
          plot_background=element_rect(color='black', fill='#e6e6e6', size=1))